# KAIST Food Recommender
Project for [CS372: NLP with Python]

Participator: Seonghun Choi, Samad Tajrian


## Initial Setting

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer
import random

import string
import json
import re

In [ ]:
dataset_path = "/content/drive/Shareddrives/NLP/dataset/"
%cd "/content/drive/Shareddrives/NLP/"
!ls

/content/drive/Shareddrives/NLP
 dataset		      'KAIST Food Recommender.ipynb'	   model
'Final Presentation.gslides'   kfr_dataset_wordnet_mapping.ipynb


## Load Dataset

In [ ]:
class DatasetLoader:
    def __init__(self, dir_path, file_list):
        self.dir = dir_path
        self.file_list = file_list

    def load_all(self):
        dataset = []
        for file_name in self.file_list:
            full_path = self.dir + file_name
            with open(full_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if isinstance(data, list):
                    dataset.extend(data)
                else:
                    dataset.append(data)  # case of single dictionary
        return dataset


In [ ]:
file_list = [
    "bears_taco_menu_with_synset.json",
    "byeoli_dali_menu_with_synset.json",
    "campus_toast_menu_with_synset.json",
    "insang_menu_with_synset.json",
    "jesoon_menu_with_synset.json",
    "little_hanoi_menu_with_synset.json",
    "onigiri_and_lee_gyudong_menu_with_synset.json",
    "pulbitmaru_menu_with_synset.json",
    "rolling_pasta_menu_with_synset.json",
    "subway_15cm_menu_with_synset.json",
    "the_big_lunch_box_menu_full_with_synset.json",
    "well_chai_menu_with_synset.json",
    "yeokjeon_menu_with_synset.json"
]

data_loader = DatasetLoader(dataset_path, file_list)
dataset = data_loader.load_all()

In [ ]:
print(dataset[:10])

[{'menu_name': 'Taco', 'temperature': 'hot.a.01', 'taste': ['mouth-watering.s.01', 'piquant.s.01', 'fresh.a.01'], 'ingredients': ['tortilla.n.01', 'lettuce.n.03', 'salsa.n.01', 'meat.n.01', 'cheese.n.01'], 'nutrition': {'carbs': '40', 'proteins': '18', 'fat': '12', 'calories': '500'}, 'price': '7900', 'allergens': ['wheat.n.02', 'dairy.n.01'], 'vegetarian': False, 'halal': False, 'restaurant': {'cuisine': 'Mexican', 'restaurant_name': 'Bears Taco', 'location': 'N'}}, {'menu_name': 'Burrito', 'temperature': 'hot.a.01', 'taste': ['mouth-watering.s.01', 'full-bodied.s.01', 'piquant.s.01'], 'ingredients': ['tortilla.n.01', 'rice.n.01', 'bean.n.01', 'meat.n.01', 'cheese.n.01', 'salsa.n.01'], 'nutrition': {'carbs': '60', 'proteins': '22', 'fat': '18', 'calories': '650'}, 'price': '8500', 'allergens': ['wheat.n.02', 'dairy.n.01'], 'vegetarian': False, 'halal': False, 'restaurant': {'cuisine': 'Mexican', 'restaurant_name': 'Bears Taco', 'location': 'N'}}, {'menu_name': 'Burrito Bowl', 'tempera

## Model Implementation

In [ ]:
class KeywordProcessor:
    def __init__(self):
        self.custom_stopwords = {
            "want", "need", "get", "look", "looking", "like",
            "would", "something", "maybe", "prefer", "think",
            "feel", "try", "eat", "drink"
        }
        self.stop_words = set(stopwords.words('english')).union(self.custom_stopwords)
        self.lemmatizer = WordNetLemmatizer()

    def preprocess(self, user_input):
        # Tokenize & lowercase
        tokens = word_tokenize(user_input.lower())

        # Remove stopwords and punctuation
        filtered_tokens = [
            word for word in tokens
            if word not in self.stop_words and word not in string.punctuation
        ]

        # POS tagging
        tagged_tokens = pos_tag(filtered_tokens)

        # Keep only nouns and adjectives
        relevant_tokens = [
            (word, tag) for word, tag in tagged_tokens
            if tag.startswith('JJ') or tag.startswith('NN')
        ]

        # Lemmatization
        lemmatized = [
            self.lemmatizer.lemmatize(word, pos='a' if tag.startswith('JJ') else 'n')
            for word, tag in relevant_tokens
        ]

        # Remove duplicates and return
        return list(set(lemmatized))

    def get_synonyms(self, word):
        synonyms = set()
        for syn in wn.synsets(word):
            for lemma in syn.lemmas():
                synonyms.add(lemma.name().lower())
        return synonyms

    def get_best_synset(self, word, pos=None):
        synsets = wn.synsets(word, pos=pos)
        if synsets:
            return synsets[0]
        return None



    def remove_unwanted_ingredients(self, sentence):

        # Count occurrences of unwanted keywords

        words = word_tokenize(sentence.lower())
        unwanted_keywords = {'no', 'without', 'but no'}

        unwanted_count = sum(1 for word in words if word in unwanted_keywords)
        print(f"Unwanted keywords found: {unwanted_count}")


        # Enhanced negation pattern to capture entire list after negation
        pattern = r'\b(no|without|but no|allergic to)\b((\s+\w+[,\s]*)+)(?=(?:\s+(but|or|so|because|and)\b|[.;]|$))'

        # Replace each matched unwanted ingredient phrase with the negation keyword only (or nothing)
        cleaned = re.sub(pattern, r'\1', sentence, flags=re.IGNORECASE)

        # Optional cleanup: remove leftover dangling punctuation
        cleaned = re.sub(r'\s+[.,;]', '', cleaned)
        cleaned = re.sub(r'\s+', ' ', cleaned).strip()

        return cleaned


In [ ]:
class KeywordScorer:
    def __init__(self, keywords, dataset):
        self.keywords = keywords
        self.keywords_synsets = self._to_synsets(keywords)
        self.dataset = dataset

    def _to_synsets(self, keyword_list):
        synsets = []
        for k in keyword_list:
            try:
                syn = wn.synsets(k)[0]
                synsets.append(syn)
            except:
                continue
        return synsets

    def _to_synset(self, synset_str):
        try:
            return wn.synset(synset_str)
        except:
            return None

    def _compare_feature_with_keywords(self, feature_values):
        score = 0.0
        for f_str in feature_values:
            f_syn = self._to_synset(f_str)
            if f_syn is None:
                continue
            similarities = [f_syn.wup_similarity(k_syn) or 0 for k_syn in self.keywords_synsets]
            score += max(similarities, default=0)
        return score

    def score(self):
        scored_items = []

        for item in self.dataset:
            total_score = 0.0
            for feature_name, feature_values in item.items():
                if isinstance(feature_values, list):
                    feature_score = self._compare_feature_with_keywords(feature_values)
                    total_score += feature_score
            scored_items.append({
                "item": item,
                "score": total_score
            })

        ranked = sorted(scored_items, key=lambda x: x["score"], reverse=True)
        return ranked

    def print_ranking(self, k = 10):
        ranking = self.score()
        for entry in ranking[:k]:
            print("Restaurant:", entry["item"]["restaurant"]["restaurant_name"], "Menu: ", entry["item"]["menu_name"], "-> Score:", entry["score"])

## Execution

In [ ]:
def main():
    user_input = input("Enter some text: ")
    kp = KeywordProcessor()
    removed_unwanted = kp.remove_unwanted_ingredients(user_input)
    preprocessed_input = kp.preprocess(removed_unwanted)

    print("Keywords:", preprocessed_input)
    random.shuffle(dataset)
    ks = KeywordScorer(preprocessed_input, dataset)
    ks.print_ranking()

if __name__ == "__main__":
    main()

Enter some text: I want chicken and rice with hot sauce
Unwanted keywords found: 0
Keywords: ['chicken', 'rice', 'sauce', 'hot']
Restaurant: Bears Taco Menu:  Burrito -> Score: 7.171405228758171
Restaurant: Bears Taco Menu:  Taco -> Score: 6.1421654626762985
Restaurant: Bears Taco Menu:  Burrito Bowl -> Score: 5.963071895424837
Restaurant: Bears Taco Menu:  Quesadilla -> Score: 4.938071895424837
Restaurant: Pulbitmaru Menu:  Chicken Breast + Rice Burrito -> Score: 4.726479188166495
Restaurant: Bears Taco Menu:  Enchilada -> Score: 4.715849673202614
Restaurant: Yeokjeon Menu:  Spicy Chicken Mayo Rice -> Score: 4.701315789473684
Restaurant: Yeokjeon Menu:  Spicy Pork Rice Bowl -> Score: 4.701315789473684
Restaurant: Yeokjeon Menu:  Beef Bulgogi Kimchi Rice -> Score: 4.701315789473684
Restaurant: Yeokjeon Menu:  Tuna Mayo Rice Ball -> Score: 4.701315789473684


In [ ]:
sample_inputs = [
    "Recommend a dish with chicken but no garlic.",
    "I want something that includes tofu and mushrooms.",
    "Suggest a meal without any dairy products.",
    "Can you recommend a dish with beef but no onions?",
    "I’m allergic to peanuts. What can I eat instead?",
    "I want to eat something that includes spinach.",
    "Suggest food that excludes eggs and milk.",
    "What can I eat that has rice but no seafood?",
    "Recommend a meal with only vegetables—no meat.",
    "I want a dessert without any sugar.",
    "Suggest a spicy dish with chili peppers but no soy sauce.",
    "What can I eat that includes salmon and lemon?",
    "Recommend a recipe without gluten or nuts.",
    "I want to try something with kimchi but no pork.",
    "Suggest a snack with cheese but no flour.",
    "I want food with shrimp but no shellfish broth.",
    "Recommend something that has beef and potatoes.",
    "I don’t want any tomatoes. What are my options?",
    "Give me a meal with lentils but no garlic or onion.",
    "Suggest a sweet food without honey or syrup.",
    "I’m avoiding soy—what can I eat instead?",
    "I want a dish with eggplant and sesame oil.",
    "Suggest something with broccoli, but hold the cheese.",
    "Recommend something that includes carrots and chicken.",
    "No spicy ingredients—what can I eat?",
    "I want to eat something with mushrooms and tofu.",
    "Suggest a seafood dish without shellfish.",
    "Recommend something with sweet potatoes but no cinnamon.",
    "What food can I eat that excludes dairy and meat?",
    "I want a meal with rice, but no soy or gluten.",
    "Suggest something with apples but no added sugar.",
    "I want to include spinach and quinoa—any ideas?",
    "Recommend something without meat or dairy.",
    "I’m avoiding gluten—what are my options?",
    "Suggest a dish with zucchini but without onions.",
    "I want something that has eggs and rice.",
    "No sugar, no dairy—what can I eat?",
    "Recommend food with bell peppers but no vinegar.",
    "I want to include avocado—what should I make?",
    "Suggest a meal that includes tofu but no sesame.",
    "I need something with beans and corn.",
    "Recommend something without garlic or ginger.",
    "What’s a good dish with beef but no dairy?",
    "I want to include chickpeas and lemon.",
    "Suggest a meal with rice noodles and no peanuts.",
    "Recommend something that uses carrots and no oil.",
    "I want to eat eggs and spinach—any ideas?",
    "Suggest a recipe without soy, nuts, or dairy.",
    "What includes sweet corn but no salt?",
    "I want something with tomatoes and basil.",
    "Recommend a dish without nuts or legumes.",
    "Give me something with seaweed and rice.",
    "I want food that excludes sugar and soy.",
    "Suggest something with kimchi and tofu.",
    "Recommend a dish that excludes onions.",
    "I want something with apples and cinnamon.",
    "Suggest food with no artificial sweeteners.",
    "What dish can I make with lentils and carrots?",
    "Recommend a snack without nuts and seeds.",
    "I want to include garlic but not onions.",
    "Suggest something with coconut milk and no dairy.",
    "I want to eat something that includes beets.",
    "Recommend a dish with eggs and no meat.",
    "I need something with cabbage but no soy sauce.",
    "Suggest a salad with no tomatoes or cucumbers.",
    "I want a meal with pumpkin and no sugar.",
    "Recommend something with strawberries and oats.",
    "I want to exclude any animal products.",
    "Suggest a smoothie with bananas but no dairy.",
    "Recommend a vegan meal with lentils and potatoes.",
    "I want food with spinach and tomatoes.",
    "Suggest a meal with tofu and bok choy.",
    "Recommend something with no processed ingredients.",
    "I want a dish that includes chickpeas and carrots.",
    "Give me something with no added oils.",
    "Suggest food with avocado but no lemon.",
    "Recommend a stir-fry without soy sauce.",
    "I want something that includes brown rice.",
    "Suggest a meal with squash and quinoa.",
    "Recommend a soup without cream or butter.",
    "I want something with tofu and green onions.",
    "Suggest a dish with cucumber and sesame oil.",
    "Recommend food that includes basil and tomatoes.",
    "I want to exclude dairy and include mushrooms.",
    "Suggest something that has lentils and no garlic.",
    "Recommend a dessert without flour or sugar.",
    "I want food that uses oats and bananas.",
    "Suggest a breakfast with no dairy and no meat.",
    "Recommend a rice dish with no soy products.",
    "I want a sandwich with no mayo or cheese.",
    "Suggest something with blueberries and no sugar.",
    "Recommend a curry without dairy.",
    "I want something with kale and sweet potato.",
    "Suggest a dish with corn and black beans.",
    "Recommend a noodle dish with no eggs.",
    "I want to avoid sugar and oil.",
    "Suggest food that includes tomatoes but no garlic.",
    "Recommend a salad with lemon but no vinegar.",
    "I want food that uses eggplant and garlic.",
    "Suggest a rice bowl with tofu and vegetables, no sauce."
]

def evaluate(sample_inputs):
    for input in sample_inputs:
        user_input = input
        kp = KeywordProcessor()
        print(f"\ninput setences {input}")
        removed_unwanted = kp.remove_unwanted_ingredients(user_input)
        preprocessed_input = kp.preprocess(removed_unwanted)

        print("Keywords:", preprocessed_input)
        random.shuffle(dataset)
        ks = KeywordScorer(preprocessed_input, dataset)
        ks.print_ranking()

evaluate(sample_inputs)


input setences Recommend a dish with chicken but no garlic.
Unwanted keywords found: 1
Keywords: ['dish', 'chicken']
Restaurant: Bears Taco Menu:  Burrito -> Score: 4.4978453263282985
Restaurant: Bears Taco Menu:  Taco -> Score: 4.115664383775839
Restaurant: Bears Taco Menu:  Burrito Bowl -> Score: 4.033793038746599
Restaurant: Bears Taco Menu:  Quesadilla -> Score: 3.718003565062389
Restaurant: Bears Taco Menu:  Enchilada -> Score: 3.273559120617944
Restaurant: Byeoli Dali Menu:  Pork Bulgogi Roe Rice Bowl -> Score: 3.24686158354889
Restaurant: SUBWAY Menu:  Shrimp -> Score: 3.1305195922068982
Restaurant: Byeoli Dali Menu:  Spicy Pork Roe Rice Bowl -> Score: 3.104004440691747
Restaurant: Byeoli Dali Menu:  Cheese Roe Rice Bowl -> Score: 3.0291146761734993
Restaurant: Yeokjeon Menu:  Shrimp Tempura -> Score: 2.867893329580636

input setences I want something that includes tofu and mushrooms.
Unwanted keywords found: 0
Keywords: ['mushroom', 'tofu']
Restaurant: Bears Taco Menu:  Burrit